# 01 — H&M Jina CLIP embeddings (Multi-GPU & Instant Discovery)

Standalone H&M Kaggle pipeline with Multi-GPU (2x T4) acceleration, instant path lookup (0 recursion), and stable FP32 inference.

## 1. Load config and instant-resolve dataset & image paths

In [ ]:
from pathlib import Path
import os, json, random, hashlib, shutil
import numpy as np
import polars as pl

PROFILE = os.getenv("HM_PROFILE", "quick").lower()
assert PROFILE in {"quick", "full"}
WORK = Path(os.getenv("HM_WORK_DIR", "/kaggle/working/hm_v1"))
WORK.mkdir(parents=True, exist_ok=True)

# 1. Resolve the explicit Kaggle Dataset first; do not recursively scan images.
dataset_hint = Path(os.getenv("HM_DATASET_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled"))
input_dirs = ([dataset_hint] if dataset_hint.is_dir() else [])
if Path("/kaggle/input").exists():
    input_dirs += [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]
input_dirs = list(dict.fromkeys(input_dirs)) or [Path.cwd()]

# 2. Tìm hoặc tạo run_config.json
cfg_file = WORK / "run_config.json"
if not cfg_file.exists():
    for d in input_dirs:
        if (d / "run_config.json").exists():
            shutil.copy(d / "run_config.json", cfg_file)
            print(f"Copied run_config.json from {d}")
            break
    if not cfg_file.exists():
        cfg = {
            "dataset": "H&M Personalized Fashion Recommendations",
            "version": "hm_v1",
            "profile": PROFILE,
            "seed": 20260922,
            "customers": 50_000 if PROFILE == "quick" else 300_000,
            "images": None
        }
        cfg_file.write_text(json.dumps(cfg, indent=2))

cfg = json.loads(cfg_file.read_text())
SEED = cfg.get("seed", 20260922)
random.seed(SEED); np.random.seed(SEED)

# 3. Tìm thư mục images trực tiếp trong các dataset (chạy trong 0.001s)
image_dir = None
for d in input_dirs:
    if (d / "images").is_dir():
        image_dir = d / "images"
        break

if image_dir:
    cfg["images"] = str(image_dir)
    cfg_file.write_text(json.dumps(cfg, indent=2))
    print(f"✅ Image folder found: {image_dir}")
else:
    print("⚠️ Image folder not found. Image embeddings will be zero-filled.")

# 4. Đảm bảo items.parquet tồn tại
items_path = WORK / "items.parquet"
if not items_path.exists():
    for d in input_dirs:
        if (d / "items.parquet").exists():
            shutil.copy(d / "items.parquet", items_path)
            print(f"Copied items.parquet from {d}")
            break
    if not items_path.exists():
        for d in input_dirs:
            if (d / "articles.csv").exists():
                art_p = d / "articles.csv"
                items = (pl.scan_csv(art_p, schema_overrides={"article_id": pl.String})
                    .select("article_id", "prod_name", "product_type_name", "product_group_name", "graphical_appearance_name", "colour_group_name", "department_name", "section_name", "detail_desc")
                    .rename({"article_id": "item_id"}).collect(streaming=True))
                items.write_parquet(items_path, compression="zstd")
                print(f"✅ Created {items_path} ({items.height:,} items) from {art_p}")
                break

print(f"🚀 Ready! Work Dir: {WORK} | Profile: {PROFILE}")

In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir "numpy==2.0.2" "scipy==1.14.1" "scikit-learn==1.5.2" "transformers==4.51.3" "tokenizers<0.22" "huggingface_hub<1.0" "einops==0.8.1" "timm==1.0.19" "pillow==11.3.0" tqdm
print("Installed Jina-compatible Transformers/Pillow stack. Restart the Kaggle session once, then run this notebook from the model-load cell.")

## 2. Load Jina CLIP v2 Model (with Multi-GPU Support)

In [ ]:
import numpy as np
import scipy
import sklearn
print(f"Binary stack: numpy={np.__version__}, scipy={scipy.__version__}, sklearn={sklearn.__version__}")
assert np.__version__ == "2.0.2" and scipy.__version__ == "1.14.1" and sklearn.__version__ == "1.5.2", "Restart the Kaggle kernel after the install cell before loading Jina."

import torch
from transformers import AutoModel
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

MODEL_ID = "jinaai/jina-clip-v2"
DIM = 512
os.environ["HF_HOME"] = str(WORK / "hf_cache")

num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Load model instances on each available GPU for true parallel inference
models = []
if num_gpus > 0:
    for gpu_id in range(num_gpus):
        # Jina's custom image code mixes internally-created Float tensors with
        # model buffers. Keep the complete model in FP32 to avoid BF16/FP32 writes.
        m = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype="float32")
        m = m.to(device=f"cuda:{gpu_id}", dtype=torch.float32).eval()
        models.append(m)
        print(f"Loaded model instance on cuda:{gpu_id} (FP32 stable mode)")
else:
    m = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype="float32").float().eval()
    models.append(m)
    print("Loaded model on CPU")

primary_model = models[0]

## 3. Trích xuất Text Embeddings

In [ ]:
from pathlib import Path
import os
import polars as pl
# Restart-safe artifact resolution: do not rely on a path object from an earlier kernel.
requested_work = Path(os.getenv("HM_WORK_DIR", "/kaggle/working/hm_v1"))
dataset_hint = Path(os.getenv("HM_DATASET_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled"))
artifact_candidates = [requested_work / "items.parquet", Path("/kaggle/working/hm_v1/items.parquet"), dataset_hint / "items.parquet"]
items_path = next((path for path in artifact_candidates if path.is_file()), None)
if items_path is None:
    search_roots = [dataset_hint] + ([p for p in Path("/kaggle/input").iterdir() if p.is_dir()] if Path("/kaggle/input").exists() else [])
    articles = next((directory / "articles.csv" for directory in search_roots if (directory / "articles.csv").is_file()), None)
    if articles is None:
        checked = "; ".join(str(path) for path in artifact_candidates)
        raise FileNotFoundError(f"items.parquet is absent ({checked}) and no H&M articles.csv input is mounted. Run notebook 00 or attach the competition input.")
    requested_work.mkdir(parents=True, exist_ok=True)
    items_path = requested_work / "items.parquet"
    (pl.scan_csv(articles, schema_overrides={"article_id": pl.String})
       .select("article_id", "prod_name", "product_type_name", "product_group_name", "graphical_appearance_name", "colour_group_name", "department_name", "section_name", "detail_desc")
       .rename({"article_id": "item_id"}).collect(streaming=True).write_parquet(items_path, compression="zstd"))
print(f"Using items artifact: {items_path} ({items_path.stat().st_size:,} bytes)")
items = pl.read_parquet(str(items_path)).fill_null("")
texts = (
    items["prod_name"] + ". " +
    items["product_type_name"] + ". " +
    items["product_group_name"] + ". " +
    items["colour_group_name"] + ". " +
    items["department_name"] + ". " +
    items["detail_desc"]
).to_list()

BATCH_SIZE_TEXT = 64
print(f"Encoding {len(texts):,} item texts...")

def encode_text_chunk(model, text_chunk, pbar=None):
    results = []
    with torch.inference_mode():
        for start in range(0, len(text_chunk), BATCH_SIZE_TEXT):
            batch = text_chunk[start:start + BATCH_SIZE_TEXT]
            # Product metadata is corpus content. This Jina checkpoint accepts
            # only retrieval.query or None; None keeps it aligned with images.
            vec = model.encode_text(batch, task=None, truncate_dim=DIM)
            results.append(np.asarray(vec, dtype="float32"))
            if pbar:
                pbar.update(len(batch))
    return np.concatenate(results) if results else np.empty((0, DIM), dtype="float32")

with tqdm(total=len(texts), desc="Text Embeddings", unit="text") as pbar:
    if len(models) > 1 and len(texts) >= 500:
        chunks = np.array_split(texts, len(models))
        with ThreadPoolExecutor(max_workers=len(models)) as ex:
            futures = [ex.submit(encode_text_chunk, models[i], list(chunk), pbar) for i, chunk in enumerate(chunks)]
            text_vectors = np.concatenate([f.result() for f in futures])
    else:
        text_vectors = encode_text_chunk(primary_model, texts, pbar)

text_vectors /= np.maximum(np.linalg.norm(text_vectors, axis=1, keepdims=True), 1e-12)
np.save(WORK / "text_embeddings.npy", text_vectors)
print("✅ Saved text_embeddings.npy with shape:", text_vectors.shape)

## 4. Trích xuất Image Embeddings (Parallel Across 2 GPUs)

In [ ]:
from threading import Lock

text_output = WORK / "text_embeddings.npy"
if not text_output.is_file():
    raise FileNotFoundError(f"Missing {text_output}; finish the text embedding cell before starting images.")
text_check = np.load(text_output, mmap_mode="r")
if text_check.shape != (items.height, DIM) or text_check.dtype != np.float32:
    raise ValueError(f"Invalid text embedding artifact: {text_check.shape}/{text_check.dtype}; expected {(items.height, DIM)}/float32")
del text_check

# Memory-mapped output is a valid .npy file and is flushed after every batch.
# If the run is interrupted, rerunning this cell resumes non-zero rows.
image_output = WORK / "image_embeddings.npy"
expected_shape = (items.height, DIM)
if image_output.is_file():
    try:
        image_vectors = np.load(image_output, mmap_mode="r+")
        if image_vectors.shape != expected_shape or image_vectors.dtype != np.float32:
            raise ValueError(f"incompatible existing array {image_vectors.shape}/{image_vectors.dtype}")
        print(f"Resuming existing image artifact: {image_output}")
    except Exception as error:
        print(f"Existing image artifact cannot be resumed ({error}); recreating it.")
        image_vectors = np.lib.format.open_memmap(image_output, mode="w+", dtype="float32", shape=expected_shape)
        image_vectors[:] = 0; image_vectors.flush()
else:
    image_vectors = np.lib.format.open_memmap(image_output, mode="w+", dtype="float32", shape=expected_shape)
    image_vectors[:] = 0; image_vectors.flush()
write_lock = Lock()
failed_images = []
img_base = Path(cfg.get("images", "")) if cfg.get("images") else None
dataset_images = Path(os.getenv("HM_DATASET_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled")) / "images"
if not img_base or not img_base.is_dir():
    img_base = dataset_images if dataset_images.is_dir() else None

if img_base and img_base.exists():
    print(f"Scanning images from: {img_base}")
    paths = [(i, img_base / str(article)[:3] / f"{article}.jpg") for i, article in enumerate(items["item_id"].to_list())]
    completed = np.linalg.norm(np.asarray(image_vectors), axis=1) > 0
    paths = [(i, p) for i, p in paths if p.is_file() and not completed[i]]
    
    limit = int(os.environ["HM_MAX_IMAGES"]) if os.getenv("HM_MAX_IMAGES") else None
    paths = paths[:limit] if limit else paths
    print(f"Total valid images to encode: {len(paths):,}")
    model_dtypes = [{tensor.dtype for tensor in list(model.parameters()) + list(model.buffers()) if tensor.is_floating_point()} for model in models]
    if any(dtypes != {torch.float32} for dtypes in model_dtypes):
        raise TypeError(f"Models are not pure FP32: {model_dtypes}. Restart the kernel and rerun the FP32 model-load cell.")
    if paths:
        # Fail in seconds, not hours, if model/image/dtype integration is broken.
        with torch.inference_mode():
            probe = np.asarray(primary_model.encode_image([str(paths[0][1])], truncate_dim=DIM), dtype="float32")
        if probe.shape != (1, DIM) or not np.isfinite(probe).all():
            raise ValueError(f"Image preflight returned invalid output: shape={probe.shape}, finite={np.isfinite(probe).all()}")
        print(f"Preflight passed: {paths[0][1]} -> {probe.shape}/{probe.dtype}")
    
    BATCH_SIZE_IMG = 16
    
    def encode_image_chunk(model, chunk_items, pbar=None):
        succeeded = 0
        with torch.inference_mode():
            for start in range(0, len(chunk_items), BATCH_SIZE_IMG):
                batch = chunk_items[start:start + BATCH_SIZE_IMG]
                encoded = []
                try:
                    matrix = np.asarray(model.encode_image([str(path) for _, path in batch], truncate_dim=DIM), dtype="float32")
                    encoded = [(idx, row) for (idx, _), row in zip(batch, matrix)]
                except Exception as batch_error:
                    # One corrupt JPEG must not discard hours of completed work.
                    for idx, path in batch:
                        try:
                            row = np.asarray(model.encode_image([str(path)], truncate_dim=DIM), dtype="float32")[0]
                            encoded.append((idx, row))
                        except Exception as item_error:
                            with write_lock:
                                failed_images.append({"item_index": int(idx), "path": str(path), "error": repr(item_error), "batch_error": repr(batch_error)})
                if encoded:
                    indices = [idx for idx, _ in encoded]
                    matrix = np.stack([row for _, row in encoded]).astype("float32")
                    matrix /= np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12)
                    with write_lock:
                        image_vectors[indices] = matrix
                        image_vectors.flush()
                    succeeded += len(indices)
                if pbar:
                    pbar.update(len(batch))
        return succeeded
    
    with tqdm(total=len(paths), desc="Image Embeddings", unit="img") as pbar:
        if len(models) > 1 and len(paths) >= 200:
            chunks = [paths[gpu_id::len(models)] for gpu_id in range(len(models))]
            with ThreadPoolExecutor(max_workers=len(models)) as ex:
                futures = [ex.submit(encode_image_chunk, models[i], chunk, pbar) for i, chunk in enumerate(chunks)]
                encoded_now = sum(f.result() for f in futures)
        else:
            encoded_now = encode_image_chunk(primary_model, paths, pbar)
    print(f"Encoded and checkpointed {encoded_now:,} new images.")
    if paths and encoded_now == 0:
        (WORK / "image_embedding_failures.json").write_text(json.dumps(failed_images, indent=2))
        raise RuntimeError("All image encodes failed; inspect image_embedding_failures.json before retrying.")
else:
    raise FileNotFoundError(f"Image folder not found. Checked cfg={cfg.get('images')} and {dataset_images}. Set HM_DATASET_DIR correctly before starting the long run.")

image_vectors.flush()
if failed_images:
    (WORK / "image_embedding_failures.json").write_text(json.dumps(failed_images, indent=2))
coverage = float((np.linalg.norm(image_vectors, axis=1) > 0).mean())
print(f"✅ Saved image_embeddings.npy with shape {image_vectors.shape} (Image coverage: {coverage * 100:.1f}%)")

manifest = {
    "model": MODEL_ID,
    "truncate_dim": DIM,
    "text_file": "text_embeddings.npy",
    "image_file": "image_embeddings.npy",
    "items": items.height,
    "image_coverage": coverage,
    "hf_cache": str(WORK / "hf_cache"),
    "gpus_used": num_gpus,
    "dtype": "float32",
    "resumable_memmap": True,
    "failed_images": len(failed_images),
    "image_source": str(img_base)
}
(WORK / "embedding_manifest.json").write_text(json.dumps(manifest, indent=2))
display(manifest)

Jina CLIP v2 produces aligned text/image vectors. This notebook owns downloading, encoding and persistent artifacts; later notebooks only read its `.npy` files and manifest.